# 02 — Tokenization

Explore how text is converted to tokens, compare tokenizers, and understand cost implications.

## Setup

In [ ]:
# Install tiktoken if not available
# !pip install tiktoken

import tiktoken
import json
import re

## 1. Tokenization with tiktoken

tiktoken is OpenAI's fast BPE tokenizer. We'll use it to understand how tokenization works.

In [ ]:
# Load GPT-4 tokenizer (cl100k_base encoding)
enc = tiktoken.get_encoding("cl100k_base")

# Basic tokenization
text = "Hello, world! How are you?"
tokens = enc.encode(text)

print(f"Original text: {text}")
print(f"Token IDs: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"Decoded tokens: {[enc.decode([t]) for t in tokens]}")

In [ ]:
# Tokens are NOT always words
examples = [
    "Hello",
    "hello",
    " supercalifragilisticexpialidocious",
    "antidisestablishmentarianism",
    "COVID",
    "tokenization",
]

print(f"{'Text':<40} {'Tokens':<8} {'Token Strings'}")
print("-" * 80)
for text in examples:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"{repr(text):<40} {len(tokens):<8} {decoded}")

## 2. How BPE Works

Byte-Pair Encoding starts with individual bytes and iteratively merges the most frequent pairs.

In [ ]:
# Demonstrate BPE concept with a simple example
def bpe_demo(text, n_merges=10):
    """Simplified BPE demonstration."""
    # Start with individual characters
    tokens = list(text.encode("utf-8"))
    print(f"Step 0: {tokens} (individual bytes)")
    print(f"        {[bytes([t]) for t in tokens]}")
    
    merges = []
    for i in range(n_merges):
        # Count adjacent pairs
        pairs = {}
        for j in range(len(tokens) - 1):
            pair = (tokens[j], tokens[j + 1])
            pairs[pair] = pairs.get(pair, 0) + 1
        
        if not pairs:
            break
        
        # Find most frequent pair
        best_pair = max(pairs, key=pairs.get)
        best_count = pairs[best_pair]
        
        # Create new token
        new_token = best_pair[0] + best_pair[1]  # merge by concatenating bytes
        new_token_id = 256 + i  # new tokens start at 256
        
        merges.append((best_pair, new_token_id))
        
        # Apply merge
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and (tokens[j], tokens[j+1]) == best_pair:
                new_tokens.append(new_token_id)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        
        tokens = new_tokens
        print(f"\nStep {i+1}: Merged {best_pair} -> {new_token_id} (count: {best_count})")
        print(f"         Tokens: {tokens}")
    
    return tokens, merges

In [ ]:
# Run BPE demo on a common word
text = "aaabbbcccddd"
final_tokens, merges = bpe_demo(text, n_merges=8)

## 3. Comparing Token Counts Across Models

Different models use different tokenizers, resulting in different token counts for the same text.

In [ ]:
# Compare different OpenAI model tokenizers
encodings = {
    "gpt-3.5-turbo": tiktoken.get_encoding("cl100k_base"),
    "gpt-4": tiktoken.get_encoding("cl100k_base"),
    "gpt-4o": tiktoken.get_encoding("o200k_base"),
    "text-davinci-003": tiktoken.get_encoding("p50k_base"),
}

test_texts = [
    "Hello, how are you today?",
    "The quick brown fox jumps over the lazy dog.",
    "import numpy as np",
    "def calculate_attention(Q, K, V):",
    "Machine learning is transforming every industry.",
]

print(f"{'Text':<50} {'gpt-3.5':<10} {'gpt-4':<10} {'gpt-4o':<10} {'davinci':<10}")
print("-" * 90)

for text in test_texts:
    counts = []
    for name, enc in encodings.items():
        count = len(enc.encode(text))
        counts.append(count)
    print(f"{repr(text):<50} {counts[0]:<10} {counts[1]:<10} {counts[2]:<10} {counts[3]:<10}")

## 4. How Tokenization Affects Cost

API pricing is per token. Fewer tokens = lower cost.

In [ ]:
# Cost estimation for GPT-4
# Pricing (approximate, check current rates):
# GPT-4: $30 / 1M input tokens, $60 / 1M output tokens
# GPT-3.5-turbo: $0.50 / 1M input tokens, $1.50 / 1M output tokens

enc = tiktoken.get_encoding("cl100k_base")

pricing = {
    "gpt-4": {"input": 30.0, "output": 60.0},
    "gpt-4-turbo": {"input": 10.0, "output": 30.0},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
}

def estimate_cost(text, model, is_output=False):
    tokens = len(enc.encode(text))
    price_per_token = pricing[model]["output" if is_output else "input"] / 1_000_000
    return tokens, tokens * price_per_token

# Example: A typical API call
system_prompt = "You are a helpful assistant that answers questions about AI."
user_message = "Explain what a transformer is in simple terms."
expected_response = "A transformer is a type of neural network architecture that uses self-attention to process sequences of data. It was introduced in the paper 'Attention Is All You Need' and is the foundation of modern language models like GPT and BERT."

print("Cost Analysis for a Typical API Call")
print("=" * 50)

total_input = system_prompt + user_message
input_tokens, input_cost = estimate_cost(total_input, "gpt-4")
output_tokens, output_cost = estimate_cost(expected_response, "gpt-4", is_output=True)

print(f"Input text: {repr(total_input[:50])}...")
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"\nGPT-4 cost: ${input_cost + output_cost:.6f}")

_, cost_35 = estimate_cost(total_input, "gpt-3.5-turbo")
_, cost_35_out = estimate_cost(expected_response, "gpt-3.5-turbo", is_output=True)
print(f"GPT-3.5-turbo cost: ${cost_35 + cost_35_out:.6f}")

In [ ]:
# Scale up: What if you process 10,000 requests/day?
requests_per_day = 10_000
avg_input_tokens = input_tokens
avg_output_tokens = output_tokens

print(f"Monthly cost at {requests_per_day:,} requests/day:")
print("=" * 50)

for model, prices in pricing.items():
    daily_input_cost = requests_per_day * avg_input_tokens * prices["input"] / 1_000_000
    daily_output_cost = requests_per_day * avg_output_tokens * prices["output"] / 1_000_000
    monthly_cost = (daily_input_cost + daily_output_cost) * 30
    print(f"{model:<20} ${monthly_cost:>10,.2f}/month")

## 5. Tokenization Edge Cases

Different types of content tokenize differently.

In [ ]:
# Different content types
content_types = {
    "English text": "The quick brown fox jumps over the lazy dog.",
    "Code": "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    "JSON": '{"name": "Alice", "age": 30, "active": true}',
    "URL": "https://example.com/path/to/resource?query=value&another=123",
    "Email": "user@example.com",
    "Numbers": "123456789012345678901234567890",
    "Whitespace": "hello                    world",
    "Special chars": "!@#$%^&*()_+-={}[]|\\:\";<>?,./`~",
}

print(f"{'Content Type':<20} {'Chars':<8} {'Tokens':<8} {'Chars/Token':<12} {'Text'}")
print("-" * 100)

for label, text in content_types.items():
    tokens = enc.encode(text)
    ratio = len(text) / len(tokens)
    print(f"{label:<20} {len(text):<8} {len(tokens):<8} {ratio:<12.1f} {text[:40]}")

In [ ]:
# How whitespace is tokenized
texts = [
    "hello world",      # single space
    "hello  world",     # double space
    "hello   world",    # triple space
    "hello\tworld",     # tab
    "hello\nworld",     # newline
]

print("Whitespace tokenization:")
print("-" * 60)
for text in texts:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"{repr(text):<20} -> {len(tokens)} tokens: {decoded}")

## 6. Context Window Usage

Understanding how much of the context window your prompt uses.

In [ ]:
# Context window analysis
context_windows = {
    "gpt-3.5-turbo": 4096,
    "gpt-4": 8192,
    "gpt-4-turbo": 128000,
    "gpt-4o": 128000,
}

# Example: System prompt + conversation history
system_prompt = "You are a helpful coding assistant. You provide clear, concise explanations and code examples."
conversation = """
User: How do I read a file in Python?
Assistant: You can read a file using:

```python
with open('file.txt', 'r') as f:
    content = f.read()
```

Or line by line:

```python
with open('file.txt', 'r') as f:
    for line in f:
        print(line.strip())
```

User: What about writing to a file?

Assistant: Use `open()` in write mode:

```python
with open('output.txt', 'w') as f:
    f.write('Hello, World!')
```

The `w` mode overwrites. Use `a` to append instead.
"""

full_prompt = system_prompt + conversation
total_tokens = len(enc.encode(full_prompt))

print(f"Prompt analysis:")
print(f"System prompt: {len(enc.encode(system_prompt))} tokens")
print(f"Conversation: {len(enc.encode(conversation))} tokens")
print(f"Total: {total_tokens} tokens")
print()

for model, window in context_windows.items():
    remaining = window - total_tokens
    pct_used = (total_tokens / window) * 100
    print(f"{model:<20} Window: {window:>6} | Used: {pct_used:>5.1f}% | Remaining: {remaining} tokens")

## Exercises

1. Tokenize a paragraph of text and find the 3 most common tokens.
2. Write a function that estimates the cost of a conversation with multiple turns.
3. Test how tokenization handles different languages (e.g., Chinese, Arabic, emoji).